In [3]:
import chromadb

client = chromadb.Client()

print("ChromaDB version:", chromadb.__version__)
print("Client created successfully!")

ChromaDB version: 1.5.9
Client created successfully!


In [4]:
collection = client.get_or_create_collection(
    name="robotics_documents",
    metadata={"hnsw:space": "cosine"}
)

print("Collection created:", collection.name)

Collection created: robotics_documents


In [5]:
documents = [
    "ROS is an open-source framework used to build and control robotic applications.",
    "A robot operating system provides communication tools between different robot components.",
    "LiDAR uses laser pulses to measure distances and create maps of the environment.",
    "Cameras provide visual information that robots can use for object detection and navigation.",
    "Radar uses radio waves to detect objects and estimate their distance and velocity.",
    "Ultrasonic sensors use sound waves to measure the distance to nearby objects.",
    "PID control uses proportional, integral, and derivative terms to reduce control error.",
    "The proportional term responds to the current error in a control system.",
    "The integral term accumulates past errors and helps remove steady-state error.",
    "The derivative term responds to the rate of change of error and can reduce overshoot.",
    "SLAM allows a robot to build a map while simultaneously estimating its position.",
    "A* is a path-planning algorithm that finds efficient routes using a cost and heuristic.",
    "RRT is a sampling-based algorithm commonly used for motion planning in robotics.",
    "Computer vision enables robots to understand and process information from images.",
    "Machine learning allows robots to learn patterns from data and make predictions.",
    "Deep learning uses neural networks with multiple layers to learn complex patterns.",
    "Sensor fusion combines information from multiple sensors to improve perception.",
    "Kalman filtering estimates the state of a system from noisy sensor measurements.",
    "Autonomous vehicles use sensors, perception, planning, and control to navigate safely.",
    "Obstacle avoidance enables robots to detect obstacles and select collision-free paths."
]

ids = [f"robotics_doc_{i+1}" for i in range(len(documents))]

collection.add(
    documents=documents,
    ids=ids
)

print("Documents added:", collection.count())

C:\Users\ughan\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:19<00:00, 4.20MiB/s]


Documents added: 20


In [6]:
query = "How does a robot avoid obstacles?"

results = collection.query(
    query_texts=[query],
    n_results=3
)

print("Query:", query)
print("\nTop 3 results:")

for i, doc in enumerate(results["documents"][0], start=1):
    print(f"{i}. {doc}")

Query: How does a robot avoid obstacles?

Top 3 results:
1. Obstacle avoidance enables robots to detect obstacles and select collision-free paths.
2. Autonomous vehicles use sensors, perception, planning, and control to navigate safely.
3. A robot operating system provides communication tools between different robot components.


In [7]:
filtered_collection = client.get_or_create_collection(
    name="robotics_documents_filtered",
    metadata={"hnsw:space": "cosine"}
)

categories = [
    "ROS", "ROS", "Sensors", "Sensors", "Sensors",
    "Control", "Control", "Control", "Control", "Planning",
    "Planning", "Planning", "AI", "AI", "AI",
    "AI", "Sensors", "Sensors", "Autonomy", "Planning"
]

filtered_collection.add(
    documents=documents,
    ids=[f"filtered_doc_{i+1}" for i in range(len(documents))],
    metadatas=[{"category": category} for category in categories]
)

print("Filtered collection documents:", filtered_collection.count())

Filtered collection documents: 20


In [8]:
results = filtered_collection.query(
    query_texts=["How do robots detect objects?"],
    n_results=3,
    where={"category": "Sensors"}
)

print("Query: How do robots detect objects?")
print("\nTop Sensor results:")

for i, doc in enumerate(results["documents"][0], start=1):
    print(f"{i}. {doc}")

Query: How do robots detect objects?

Top Sensor results:
1. Cameras provide visual information that robots can use for object detection and navigation.
2. Radar uses radio waves to detect objects and estimate their distance and velocity.
3. Sensor fusion combines information from multiple sensors to improve perception.


In [9]:
print("Manual verification:")
print("✓ Similarity search returned relevant robotics documents.")
print("✓ Metadata filter restricted results to category='Sensors'.")
print("✓ All returned results are sensor-related.")

Manual verification:
✓ Similarity search returned relevant robotics documents.
✓ Metadata filter restricted results to category='Sensors'.
✓ All returned results are sensor-related.


In [11]:
import ollama

query = "How does a robot avoid obstacles?"

rag_results = collection.query(
    query_texts=[query],
    n_results=3
)

top_chunks = rag_results["documents"][0]

print("Retrieved top-3 chunks:\n")

for i, chunk in enumerate(top_chunks, start=1):
    print(f"{i}. {chunk}")

Retrieved top-3 chunks:

1. Obstacle avoidance enables robots to detect obstacles and select collision-free paths.
2. Autonomous vehicles use sensors, perception, planning, and control to navigate safely.
3. A robot operating system provides communication tools between different robot components.


In [12]:
context = "\n".join(
    f"- {chunk}" for chunk in top_chunks
)

prompt = f"""
Answer the question using only the context below.

Context:
{context}

Question:
{query}

Give a concise answer in 2-3 sentences.
"""

response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {
            "role": "system",
            "content": "You are a robotics assistant. Answer only from the provided context."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]
)

answer = response["message"]["content"]

print("RAG Answer:")
print(answer)

RAG Answer:
A robot avoids obstacles by using sensors to detect their presence and then selecting a collision-free path using planning and control algorithms. This process is often assisted by advanced perception systems that enable the robot to interpret and understand its environment. The robot operating system also plays a crucial role in facilitating communication between the different components involved in obstacle avoidance.


In [13]:
print("RAG verification:")
print("✓ Query was embedded and searched in ChromaDB.")
print("✓ Top-3 relevant chunks were retrieved.")
print("✓ Retrieved chunks were passed to Ollama.")
print("✓ Ollama generated an answer using the retrieved context.")
print("✓ ChromaDB + Ollama RAG flow completed successfully.")

RAG verification:
✓ Query was embedded and searched in ChromaDB.
✓ Top-3 relevant chunks were retrieved.
✓ Retrieved chunks were passed to Ollama.
✓ Ollama generated an answer using the retrieved context.
✓ ChromaDB + Ollama RAG flow completed successfully.


In [ ]:
## W5D3 Summary — ChromaDB & RAG

### Completed Tasks

- Installed and configured ChromaDB.
- Created a ChromaDB collection using cosine similarity.
- Added 20 robotics/AI documents with automatically generated embeddings.
- Performed similarity search and retrieved the top-3 relevant documents.
- Added metadata to documents and tested metadata filtering.
- Manually verified the retrieved results.
- Connected ChromaDB with Ollama.
- Retrieved the top-3 chunks for a user query.
- Passed the retrieved context to Llama 3.2:3b.
- Generated a context-based answer using the RAG pipeline.

### RAG Pipeline

**User Query → ChromaDB → Top-3 Retrieved Chunks → Ollama → Final Answer**

### Key Learning

Learned how vector databases store embedded documents and retrieve semantically similar information using cosine similarity. Also learned how retrieved context can be combined with a local LLM to build a simple Retrieval-Augmented Generation (RAG) pipeline.